In [ ]:
import modal

image = (
    modal.Image.debian_slim(python_version="3.14")
    .uv_pip_install(
        "torch",
        "transformers",
        "trl",
        "peft",
        "bitsandbytes",
        "accelerate",
        "datasets",
        "scikit-learn",
        "numpy",
    )
)

app = modal.App("qlora-pubmedqa", image=image)

hf_secret = modal.Secret.from_name("huggingface")

In [ ]:
def build_dataset():
    from datasets import DatasetDict, load_dataset

    split = load_dataset("qiaojin/PubMedQA", "pqa_artificial", split="train") \
        .shuffle(seed=42).select(range(15000)) \
        .train_test_split(test_size=0.1)

    ds = DatasetDict({
        "train": split["train"],
        "val": split["test"],
        "test": load_dataset("qiaojin/PubMedQA", "pqa_labeled", split="train")
    })

    ds["test"] = ds["test"].filter(lambda x: x["final_decision"] != "maybe")
    ds = ds.map(
        lambda x: {
            "prompt": f"""Context:
            {'\n'.join([f"{label}: {context}" for label, context in zip(x["context"]["labels"], x["context"]["contexts"])])}
        Question: {x["question"]}
        Decision:""",
            "completion": f" {x["final_decision"]}"
        },
        remove_columns=["question", "context", "final_decision", "pubid", "long_answer"]
    )
    return ds


ds = build_dataset()
print(ds)
print(ds["train"].features)
print(ds["train"][:5])

In [ ]:
from collections import Counter

print("Train Distribution:", Counter(ds["train"]["completion"]))
print("Val Distribution:", Counter(ds["val"]["completion"]))
print("Test Distribution:", Counter(ds["test"]["completion"]))

In [ ]:
import numpy as np
import torch
from sklearn.metrics import classification_report, f1_score

MODEL_ID = "meta-llama/Llama-3.1-8B"
LABELS = [" yes", " no"]


@torch.no_grad()
def evaluate(model, tokenizer, test_ds):
    label_ids = [tokenizer(label, add_special_tokens=False)["input_ids"] for label in LABELS]
    model.eval()
    y_pred, y_true = [], []

    for sample in test_ds:
        prompt_ids = tokenizer(sample["prompt"], return_tensors="pt")["input_ids"].to(model.device)
        start = prompt_ids.shape[1] - 1
        scores = []
        for ids in label_ids:
            cand = torch.tensor([ids], device=model.device)
            seq = torch.cat([prompt_ids, cand], dim=1)
            logp = model(seq).logits[0, start:].log_softmax(-1)
            scores.append(sum(logp[k, tok].item() for k, tok in enumerate(ids)))
        y_pred.append(int(np.argmax(scores)))
        y_true.append(LABELS.index(sample["completion"]))

    macro = f1_score(y_true, y_pred, average="macro")
    print(classification_report(y_true, y_pred, target_names=LABELS))
    return macro

In [ ]:
@app.function(gpu="A100", secrets=[hf_secret], timeout=60 * 60)
def evaluate_baseline():
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer

    ds = build_dataset()
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, dtype=torch.bfloat16, device_map="cuda"
    )
    return evaluate(model, tokenizer, ds["test"])

In [ ]:
with app.run():
    baseline_macro = evaluate_baseline.remote()
print("Baseline macro F1:", baseline_macro)

In [ ]:
@app.function(gpu="A100", secrets=[hf_secret], timeout=60 * 60 * 6)
def train_qlora():
    import torch
    from peft import LoraConfig
    from transformers import (
        AutoModelForCausalLM,
        AutoTokenizer,
        BitsAndBytesConfig,
        EarlyStoppingCallback,
    )
    from trl import SFTConfig, SFTTrainer

    ds = build_dataset()
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    nf4_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, quantization_config=nf4_config, device_map="cuda"
    )

    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
        task_type="CAUSAL_LM",
        lora_dropout=0.05,
    )
    trainer = SFTTrainer(
        model=model,
        peft_config=lora_config,
        args=SFTConfig(
            completion_only_loss=True,
            output_dir="/tmp/results",
            seed=42,
            num_train_epochs=4,
            metric_for_best_model="eval_loss",
            load_best_model_at_end=True,
            greater_is_better=False,
            optim="paged_adamw_32bit",
            save_total_limit=2,
            eval_strategy="steps",
            eval_steps=200,
            save_strategy="steps",
            save_steps=200,
        ),
        train_dataset=ds["train"],
        eval_dataset=ds["val"],
        processing_class=tokenizer,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=3, early_stopping_threshold=0.0)],
    )
    trainer.train()
    return evaluate(model, tokenizer, ds["test"])

In [ ]:
with app.run():
    finetuned_macro = train_qlora.remote()
print("Fine-tuned macro F1:", finetuned_macro)